[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Error estándar e intervalos de confianza.

**py401 · EDA Práctico** · Pythonista® · Instructor: José Luis Chiquete · Módulo III: *Inferencia estadística*

> 📝 **Versión del alumno.** Las celdas marcadas con `# TU CÓDIGO AQUÍ` y las de *Tu respuesta* son para ti.

En este *notebook* se calcularán el error estándar e intervalos de confianza para medias y proporciones, y se revisará cuándo se puede confiar en la fórmula *t*.

## 🧭 Cómo usar este *notebook*.

* Ejecuta las celdas **en orden** (`Shift + Enter`).
* 🟢 **Ejercicio básico:** lo que todos deben dominar.
* 🔴 **Reto (opcional):** para quien termina antes o quiere profundizar. Si no da tiempo en clase, se puede resolver como práctica.
* Las celdas `# TU CÓDIGO AQUÍ` y *✍️ Tu respuesta* son para ti.

## 📚 Referencias.

* La generación de números aleatorios con `np.random.default_rng` se explica en [py311 · 04 Arreglos con contenido aleatorio](https://github.com/PythonistaMX/py311/blob/main/04_arreglos_con_contenido_aleatorio.ipynb) y los resúmenes con *NumPy* en [py311 · 07 Gestión y análisis de datos de *NumPy*](https://github.com/PythonistaMX/py311/blob/main/07_gestion_y_analisis_de_datos_de_numpy.ipynb).
* py311 no cubre inferencia, así que la referencia de la API es la documentación oficial: [*SciPy* · `scipy.stats`](https://docs.scipy.org/doc/scipy/reference/stats.html), [*SciPy* · `bootstrap`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html) y [*statsmodels* · `proportion_confint`](https://www.statsmodels.org/stable/generated/statsmodels.stats.proportion.proportion_confint.html).

> ⚠️ **Nota:** py311 se escribió para *pandas* 2 y este curso usa *pandas* 3; si algo se ve distinto, se sigue lo que muestra este *notebook*.

### Preparación del entorno.

* La siguiente celda importará las bibliotecas, incluidos `stats` de *SciPy* y las funciones de *statsmodels* que se usarán, y definirá `DATA` y la semilla `rng`.

In [ ]:
import os
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

DATA = next(os.path.join(p, "data") for p in (os.curdir, os.pardir)
            if os.path.isfile(os.path.join(p, "data", "penguins.csv")))
pd.set_option("display.max_columns", 20)
sns.set_theme(style="whitegrid")
rng = np.random.default_rng(401)
from scipy import stats
from statsmodels.stats.proportion import proportion_confint
from statsmodels.stats.weightstats import DescrStatsW

**¿Por qué este paso?** Se parte del archivo limpio de `09_auditoria_de_calidad.ipynb` para no repetir la limpieza; si no existe (por ejemplo, en un *Codespace* nuevo), se regenera con los mismos pasos mínimos. `titanic` se usará para proporciones y para un ejemplo de datos muy asimétricos.

* La siguiente celda leerá `penguins_limpio.csv` (o lo regenerará desde `penguins.csv` si no existe) y `titanic.csv`.

In [ ]:
ruta_limpio = os.path.join(DATA, "penguins_limpio.csv")
medidas = ["bill_length_mm", "bill_depth_mm", "flipper_length_mm", "body_mass_g"]
if not os.path.isfile(ruta_limpio):
    (
        pd.read_csv(os.path.join(DATA, "penguins.csv"))
        .assign(sex=lambda d: d["sex"].str.title())
        .dropna(subset=medidas, how="all")
        .to_csv(ruta_limpio, index=False)
    )
pinguinos = pd.read_csv(ruta_limpio)
titanic = pd.read_csv(os.path.join(DATA, "titanic.csv"))
pinguinos.shape, titanic.shape

### Objetos de los *notebooks* anteriores.

Este *notebook* es independiente: la siguiente celda vuelve a crear lo que se necesita de los *notebooks* anteriores.

* La siguiente celda definirá `poblacion` (la masa de todos los pingüinos, usada como población de juguete) y `tarifas` (las tarifas del Titanic), que se usaron en `16_distribucion_muestral.ipynb`.

In [ ]:
poblacion = pinguinos["body_mass_g"].dropna().to_numpy()
tarifas = titanic["fare"].to_numpy()
len(poblacion), len(tarifas)

---
## 1. Error estándar e intervalos de confianza (25 min).

En la práctica solo hay **una** muestra, así que el error estándar se **estima** con la desviación estándar de esa muestra: se divide entre la raíz cuadrada de *n*, como en `16_distribucion_muestral.ipynb`.

Un **intervalo de confianza** (IC) del 95% para la media es un rango de valores plausibles para la media de la población. La idea que conviene recordar, la única cuenta de este *notebook*, es:

**IC del 95% ≈ media de la muestra ± 2 errores estándar**

El "2" exacto lo da la distribución *t* de Student (≈ 1.98 con 123 datos; un poco más con muestras pequeñas). *SciPy* y *statsmodels* lo calculan solos.

**Ejemplo cotidiano:** una encuesta dice "42% a favor, ± 3 puntos". Los 3 puntos son, aproximadamente, dos errores estándar: el resultado real probablemente está entre 39% y 45%.

**Ejemplo:** desde aquí se vuelve a la situación real: los Gentoo medidos son una **muestra** de la población de Gentoo.

**Concepto: de la distribución muestral al intervalo.** Por el TCL, en 95% de las muestras la media de la muestra cae a menos de unos 2 errores estándar de la media de la población. Si eso es cierto, también es cierto al revés: la media de la población está a menos de unos 2 errores estándar de la media de la muestra. De ahí sale el intervalo.

**Ejemplo cotidiano:** si un perro casi siempre se queda a menos de 10 m de su dueño, al ver al perro se puede decir que el dueño "casi seguro" está a menos de 10 m de él.

Como la desviación estándar de la población no se conoce y se estima con la de la muestra, se usa la *t* de Student, que da un número un poco mayor que 1.96 para compensar esa incertidumbre extra.

> 💡 **La red y el pez: qué es «95%».** Imagina que la media verdadera de la población es un pez quieto en un estanque y que cada intervalo es una red que se lanza a ciegas. El pez no se mueve; las redes caen en lugares distintos en cada muestra. Un método «del 95%» es uno cuyas redes atrapan al pez en 95 de cada 100 lanzamientos. Por eso el 95% describe la **tasa de éxito del método**, no una probabilidad sobre un intervalo ya calculado: ese intervalo concreto atrapó al pez o no, y no se sabe cuál de las dos cosas ocurrió.

* La siguiente celda calculará el IC del 95% para la masa media de los Gentoo paso a paso: media, error estándar, valor de la *t* y los dos límites.

In [ ]:
masa_gentoo = pinguinos.loc[pinguinos["species"] == "Gentoo", "body_mass_g"].dropna()
n = len(masa_gentoo)
media = masa_gentoo.mean()
error_estandar = masa_gentoo.std(ddof=1) / np.sqrt(n)
t_critico = stats.t.ppf(0.975, df=n - 1)
n, round(float(media), 1), round(float(error_estandar), 1), np.round([media - t_critico * error_estandar, media + t_critico * error_estandar], 1)

**Interpretación:** con 123 Gentoo, la masa media es 5,076 g con EE de 45.5 g; el IC del 95% va de 4,986 a 5,166 g. El margen (≈ 90 g) es dos errores estándar. Ese margen es mucho menor que la dispersión de los individuos (≈ 500 g), porque describe la incertidumbre sobre **la media**, no sobre cada pingüino.

**¿De qué depende el ancho de un intervalo?** De tres cosas:
* **La variabilidad de los datos:** si los individuos varían más, la media de la muestra es menos estable y el intervalo es más ancho.
* **El tamaño de la muestra:** más datos, intervalo más angosto. Cuadruplicar *n* reduce el ancho a la mitad (con 492 Gentoo en vez de 123, el ancho de 180 g bajaría a unos 89 g).
* **El nivel de confianza:** pedir más confianza obliga a un intervalo más ancho. Con los mismos 123 Gentoo, el IC del 90% va de 5,001 a 5,151 g (ancho de unos 151 g), el del 95% de 4,986 a 5,166 g (180 g) y el del 99% de 4,957 a 5,195 g (unos 238 g).

> ⚠️ **Error común:** el IC del 95% **no** dice que el 95% de los pingüinos pese entre 4,986 y 5,166 g. De hecho, solo 14 de los 123 Gentoo (11%) caen dentro de ese rango, porque describe dónde puede estar **la media**, no dónde están **los individuos**. Para describir a los individuos sirve más un rango como la media ± 2 desviaciones estándar (de 4,068 a 6,084 g), que contiene al 98% de estos Gentoo.

> ⚠️ **Otro error común:** el intervalo solo recoge el azar del muestreo. No corrige una muestra sesgada (por ejemplo, si los pingüinos medidos no representan bien a todos los Gentoo): el cálculo sale bien hecho, pero apunta al lugar equivocado.

* La siguiente celda obtendrá el mismo intervalo con `stats.t.interval` de *SciPy* y con `DescrStatsW` de *statsmodels*.

In [ ]:
con_scipy = stats.t.interval(0.95, df=n - 1, loc=media, scale=stats.sem(masa_gentoo))
con_statsmodels = DescrStatsW(masa_gentoo).tconfint_mean(alpha=0.05)
np.round(con_scipy, 1), np.round(con_statsmodels, 1)

**Interpretación:** las dos bibliotecas reproducen exactamente el cálculo a mano. En un informe se usa una de ellas (menos riesgo de errores de escritura), pero conviene saber qué fórmula aplican. `stats.sem` usa `ddof=1` por defecto.

### 1.1 Intervalos para una proporción.

Para una proporción (casos que cumplen entre el total) hay varios métodos (`proportion_confint(k, n, method=...)`, con `k` casos de `n`):

| Método | `method=` | Comentario |
|---|---|---|
| Wald | `"normal"` | La proporción ± 1.96 errores estándar. Simple, pero falla con pocos datos o con proporciones cerca de 0 o de 1 |
| Wilson | `"wilson"` | Recomendado por defecto: funciona bien incluso con pocos datos |
| Clopper-Pearson | `"beta"` | "Exacto": conservador (intervalos algo más anchos) |

**Concepto:** una proporción es la media de una variable 0/1 (por ejemplo, 1 = sobrevivió, 0 = no), así que también tiene su error estándar. Es más grande cuando la proporción está cerca de 50% y más chico cuando se acerca a 0% o a 100%, y baja al aumentar *n*. Wald suma y resta 1.96 errores estándar; Wilson usa un cálculo que **nunca se sale de 0 a 1** y que corre un poco el centro hacia 50%; Clopper-Pearson usa la distribución binomial exacta.

* La siguiente celda calculará los tres intervalos para la tasa de supervivencia del Titanic.

In [ ]:
sobrevivientes, pasajeros = int(titanic["survived"].sum()), len(titanic)
pd.DataFrame({metodo: proportion_confint(sobrevivientes, pasajeros, alpha=0.05, method=metodo)
              for metodo in ["normal", "wilson", "beta"]}, index=["inferior", "superior"]).T.round(3)

Con 891 pasajeros los tres métodos coinciden (≈ 35% a 42%). La decisión importa con muestras pequeñas.

> 💡 **El margen de error de las encuestas** es este mismo cálculo. Con unas 1,000 personas y una proporción cercana a 50%, 1.96 × raíz cuadrada de (0.5 × 0.5 ÷ 1,000) da unos 3.1 puntos: de ahí el «± 3 puntos». Con 400 personas el margen sube a unos 5 puntos, y con 2,500 baja a unos 2. Ojo: ese margen solo recoge el azar del muestreo; no cubre los sesgos (quién contesta, cómo se pregunta).

**Decisión: ¿Wald o Wilson con pocos datos?** Con 2 éxitos en 30 casos, la fórmula de Wald da un límite inferior **negativo**, imposible para una proporción (`statsmodels` lo recorta a 0, lo que oculta el problema).

* La siguiente celda calculará Wald a mano y Wilson con *statsmodels* para 2 éxitos en 30.

In [ ]:
p_gorro = 2 / 30
margen = 1.96 * np.sqrt(p_gorro * (1 - p_gorro) / 30)
{"Wald a mano": (round(float(p_gorro - margen), 3), round(float(p_gorro + margen), 3)),
 "Wilson": tuple(round(float(v), 3) for v in proportion_confint(2, 30, method="wilson"))}

**Interpretación:** Wald da (−0.023, 0.156), con un límite inferior imposible; Wilson da (0.018, 0.213), asimétrico alrededor de la proporción observada (0.067) como corresponde a una proporción cercana a 0. **Decisión:** usar Wilson por defecto, sobre todo si hay menos de 10 éxitos o menos de 10 fracasos (aquí, solo 2 éxitos).

### 1.2 Consecuencia: ¿cuándo se puede confiar en la fórmula *t*?

El "95%" es una promesa sobre el **procedimiento**: si se repitiera el muestreo, 95% de los intervalos capturarían la media verdadera de la población. Esa promesa depende de supuestos. Se simulará la **cobertura real** del IC *t* con dos poblaciones: la masa de los pingüinos (sin colas largas) y la tarifa del Titanic (muy asimétrica, con atípicos).

**Concepto: cobertura.** Es la proporción de intervalos, sobre muchas muestras repetidas, que contienen el parámetro verdadero. Un método "del 95%" que en realidad cubre 80% es engañoso. La forma más práctica de medirla es por simulación, cuando se conoce la población.

* La siguiente celda definirá `cobertura_t`, que saca muchas muestras, calcula el IC *t* de cada una y cuenta qué fracción contiene la media poblacional.

In [ ]:
def cobertura_t(poblacion, n, repeticiones=2000, generador=rng):
    muestras = generador.choice(poblacion, size=(repeticiones, n))
    medias = muestras.mean(axis=1)
    errores = muestras.std(axis=1, ddof=1) / np.sqrt(n)
    t_critico = stats.t.ppf(0.975, df=n - 1)
    contiene = (medias - t_critico * errores <= poblacion.mean()) & (poblacion.mean() <= medias + t_critico * errores)
    return contiene.mean()

* La siguiente celda calculará la cobertura con muestras de 10, 30 y 100 en ambas poblaciones (`tarifas` se definió arriba, en este mismo *notebook*).

In [ ]:
pd.DataFrame({n: {"masa (pingüinos)": cobertura_t(poblacion, n), "tarifa (Titanic)": cobertura_t(tarifas, n)}
              for n in [10, 30, 100]}).round(3)

Con la masa, la cobertura es ≈ 95% incluso con *n* = 10 (los decimales de la tabla cambian alrededor de un punto porcentual con otra semilla; lo que importa es la diferencia entre ≈ 95% y ≈ 80%). Con la tarifa, un IC "del 95%" con *n* = 10 solo captura la media en torno a 80% de las veces, y aún con *n* = 100 se queda debajo de 95%. **Consecuencia:** con datos muy asimétricos y muestras pequeñas, el intervalo es demasiado estrecho y transmite más seguridad de la que tiene. Conviene reportar la mediana con un IC *bootstrap* (ojo: ese intervalo sería el de la **mediana**, no el de la media), transformar (logaritmo) o, al menos, advertirlo.

📖 **Para profundizar (opcional).** *Las fórmulas de la sección 1.* Con $\bar{x}$ la media de la muestra, $s$ su desviación estándar y $t_{0.975,\,n-1}$ el cuantil de la *t* de Student con $n - 1$ grados de libertad:

$$IC_{95\%} = \bar{x} \pm t_{0.975,\,n-1}\,\frac{s}{\sqrt{n}}$$

Para una proporción $\hat{p} = k / n$, el error estándar es $EE(\hat{p}) = \sqrt{\hat{p}(1-\hat{p})/n}$ y el intervalo de Wald es $\hat{p} \pm 1.96\,EE$. Wilson se obtiene "invirtiendo" la prueba *z*: incluye todos los valores de la proporción que esa prueba no rechazaría. La regla práctica de la sección 1.1 se escribe $n\hat{p} \geq 10$ y $n(1-\hat{p}) \geq 10$.

**💡 Ideas clave de la sección 1:**
* **IC del 95% ≈ media ± 2 errores estándar** (el valor exacto lo da la *t*); Wilson para proporciones.
* El 95% es una propiedad del **procedimiento**, que solo se cumple si se cumplen sus supuestos.
* Con datos muy asimétricos y pocos datos, el IC *t* cubre menos de lo prometido.
* El intervalo solo recoge el azar del muestreo; no corrige una muestra sesgada.

### 🟢 Ejercicio 1 · Supervivencia en tercera clase.

Calcula la tasa de supervivencia de los pasajeros de **tercera clase** con su IC del 95% por Wilson, y compárala con la de primera clase. ¿Se traslapan los intervalos?

**🎯 Objetivo:** calcular IC de proporciones por grupo con `proportion_confint`. **Qué se espera aprender:** a comparar grupos con su incertidumbre, no solo con sus tasas puntuales.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ

**✍️ Tu respuesta:**

_(escribe aquí)_

### 🔴 Reto 1 · Cobertura de Wald y Wilson.

Simula 5,000 muestras de tamaño 20 de una población con proporción verdadera 0.10 (`generador.binomial(20, 0.10, size=5000)`, con `generador = np.random.default_rng(401)`). Para cada muestra calcula el IC de Wald y el de Wilson y estima la cobertura de cada método. ¿Cuál cumple mejor la promesa del 95%?

**🎯 Objetivo:** medir por simulación la cobertura real de dos métodos. **Qué se espera aprender:** a verificar una afirmación metodológica ("Wilson es mejor") con datos simulados en lugar de aceptarla por autoridad.

* La siguiente celda es para tu solución.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

**✍️ Tu respuesta:**

_(escribe aquí)_

---
## ✅ Cierre.

**Práctica (en Classroom):** terminar los ejercicios 🟢 pendientes de este *notebook* y, si es posible, un reto 🔴.

**Siguiente *notebook*:** `18_bootstrap_e_interpretacion_de_intervalos.ipynb`.

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>